# Level 4 - Kidney + body + hormones
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/kidney-sim/blob/main/notebooks/04_whole_body_feedback.ipynb)

**Goal:** put all previous levels together and add the **feedback loops** that keep the body stable
(homeostasis): ADH, thirst and aldosterone.

In [ ]:
# --- Setup: run this cell first --------------------------------------------
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/kidney-sim.git"   # change if you forked the project
if "google.colab" in sys.modules and not os.path.exists("kidneysim"):
    if not os.path.exists("kidney-sim"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("kidney-sim")
for path in [".", ".."]:
    if os.path.exists(os.path.join(path, "kidneysim")):
        sys.path.insert(0, os.path.abspath(path))
        break

import numpy as np
import matplotlib.pyplot as plt
import kidneysim as ks
from kidneysim import viz2d, viz3d
from kidneysim.ui import sliders
print("kidneysim", ks.__version__, "| interactive 3D (plotly):", viz3d.HAS_PLOTLY)

## 1. Balance equations

$$\frac{dV}{dt} = \text{drink} - \text{urine} - \text{insensible loss}$$
$$\frac{dS}{dt} = 2\,(\text{salt intake} - \text{Na excreted}) \qquad \frac{dU}{dt} = G_{urea} - \text{urea excreted}$$

Plasma: $P_{Na} = S/2V$, $\;$ tonicity $= S/V + \text{glucose}$.

## 2. Feedback (hormones)

$$ADH = \text{clip}\big(0.65 + 0.08\,(\text{tonicity}-285),\,0,\,1\big)$$
$$\text{thirst} = \text{clip}\big(0.5\,(\text{tonicity}-287),\,0,\,10\big)\ \text{L/day}$$
$$\text{aldosterone} = \text{clip}\Big(1 + 30\,\frac{V_0 - V}{V_0} + 0.05\,(100 - MAP),\,0,\,3\Big)$$

Every 30 minutes the model computes GFR (Level 1), runs the nephron (Level 2) with the gradient from Level 3, and updates the body.

In [ ]:
print("available scenarios:", list(ks.coupled.SCENARIOS))
runs = {name: ks.coupled.run_scenario(name, days=3)
        for name in ["healthy", "dehydration", "drink_5_litres", "diabetes"]}
viz2d.plot_body(runs);

In [ ]:
runs = {name: ks.coupled.run_scenario(name, days=5)
        for name in ["healthy", "kidney_failure", "low_pressure", "furosemide", "salty_diet"]}
viz2d.plot_body(runs);

### Your own experiment (each run takes a few seconds)

In [ ]:
def experiment(water_intake=2.3, salt_intake=150.0, glucose=5.0, nephron_fraction=1.0, MAP_set=100.0):
    out = ks.coupled.simulate(days=3, water_intake=water_intake, salt_intake=salt_intake,
                              glucose=glucose, nephron_fraction=nephron_fraction, MAP_set=MAP_set)
    viz2d.plot_body({"your patient": out, "healthy": ks.coupled.simulate(days=3)})
    plt.show()

sliders(experiment, water_intake=(0, 6, 0.1, 2.3), salt_intake=(20, 400, 10, 150), glucose=(4, 30, 1, 5),
        nephron_fraction=(0.05, 1, 0.05, 1.0), MAP_set=(50, 160, 5, 100))

## Questions
1. When you drink 5 L, what keeps plasma Na normal? Which hormone changes?
2. Without water, ADH saturates at 1. What happens to plasma Na after that? Why is this dangerous?
3. In kidney failure, why does urea keep rising while Na stays almost normal?
4. Find one **negative feedback loop** in the model and draw it as a diagram.